# PaddleOCR — smoke test de paridad Local ↔ Colab

**No usa venv y no reinicia el kernel.**

El notebook controlador instala las versiones exactas en el runtime actual,
pero **jamás importa PaddleOCR en el kernel**. La prueba se ejecuta en un
proceso Python nuevo. Así una instalación recién hecha se prueba sin arrastrar
módulos binarios que el kernel haya cargado antes.

Para paridad con tu Docker local, el Cloud final debería fijarse al runtime
**Colab 2026.07 (Python 3.12.13)**.

Orden:
1. diagnóstico sin importar Torch/Pillow/Paddle;
2. instalación oficial e idempotente;
3. worker fresco: Paddle → CUDA → Pillow → PaddleOCR → OCR real;
4. guardar `pip freeze`.

**No ejecutes imports manuales de `torch`, `PIL`, `paddle` o `paddleocr`
antes de terminar la prueba.**

In [ ]:
import sys, platform, subprocess, pathlib, os, importlib.metadata as md

print("=== RUNTIME ===")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)
print("HOME:", pathlib.Path.home())

print("\n=== GPU ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== PAQUETES (sin importarlos) ===")
for dist in [
    "paddlepaddle-gpu",
    "paddleocr",
    "paddlex",
    "torch",
    "pillow",
    "numpy",
    "nvidia-nccl-cu12",
    "nvidia-cudnn-cu12",
]:
    try:
        print(f"{dist:22} {md.version(dist)}")
    except md.PackageNotFoundError:
        print(f"{dist:22} NO INSTALADO")

if sys.version_info[:2] != (3, 12):
    print()
    print("⚠️ Este runtime NO es el entorno de paridad esperado (Python 3.12).")
    print("Local Docker esperado: Python 3.12.13.")
    print("En Colab Cloud usaremos/pinearemos Runtime Version 2026.07.")
else:
    print("\n✅ Python 3.12: entorno apto para la prueba de paridad.")

In [ ]:
import sys, subprocess, importlib.metadata as md, os, pathlib

PADDLE_VERSION = "3.2.0"
PADDLEOCR_VERSION = "3.2.0"
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"

os.environ.setdefault("PIP_CACHE_DIR", str(pathlib.Path.home() / ".cache" / "pip"))
os.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")
os.environ.setdefault(
    "PADDLE_PDX_CACHE_HOME",
    str(pathlib.Path.home() / ".cache" / "paddlex"),
)

def version(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None

def run(cmd):
    print("\n$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

print("=== SETUP IDEMPOTENTE ===")
print("Paddle instalado:", version("paddlepaddle-gpu"))
print("PaddleOCR instalado:", version("paddleocr"))
print("Pip cache:", os.environ["PIP_CACHE_DIR"])
print("Model cache:", os.environ["PADDLE_PDX_CACHE_HOME"])

if version("paddlepaddle-gpu") != PADDLE_VERSION:
    run([
        sys.executable, "-m", "pip", "install",
        f"paddlepaddle-gpu=={PADDLE_VERSION}",
        "-i", PADDLE_INDEX,
    ])
else:
    print("✅ PaddlePaddle GPU exacto ya instalado; no se descarga de nuevo.")

if version("paddleocr") != PADDLEOCR_VERSION:
    run([
        sys.executable, "-m", "pip", "install",
        f"paddleocr=={PADDLEOCR_VERSION}",
    ])
else:
    print("✅ PaddleOCR exacto ya instalado; no se descarga de nuevo.")

print("\n=== VERSIONES DESPUÉS DE INSTALAR (sin importar) ===")
for dist in [
    "paddlepaddle-gpu",
    "paddleocr",
    "paddlex",
    "torch",
    "pillow",
    "numpy",
    "nvidia-nccl-cu12",
    "nvidia-cudnn-cu12",
]:
    print(f"{dist:22} {version(dist) or 'NO INSTALADO'}")

print("\n✅ Setup terminado.")
print("NO reinicies el kernel.")
print("La siguiente celda prueba todo en un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os

worker_path = pathlib.Path("/content/work/.colab-dev/paddle_smoke_worker.py")
worker_path.parent.mkdir(parents=True, exist_ok=True)

WORKER = '\nimport os\nimport sys\nimport json\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault(\n    "PADDLE_PDX_CACHE_HOME",\n    str(Path.home() / ".cache" / "paddlex"),\n)\n\nprint("=== WORKER FRESCO ===", flush=True)\nprint("Python:", sys.version, flush=True)\nprint("PADDLE_PDX_CACHE_HOME:", os.environ["PADDLE_PDX_CACHE_HOME"], flush=True)\n\nprint("\\n[1/5] import paddle", flush=True)\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA compiled:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda():\n    raise RuntimeError("Paddle fue instalado sin CUDA.")\nif paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve ninguna GPU.")\n\npaddle.set_device("gpu:0")\ntry:\n    print("GPU:", paddle.device.cuda.get_device_name(), flush=True)\nexcept Exception:\n    print("GPU: gpu:0", flush=True)\n\nprint("\\n[2/5] import Pillow", flush=True)\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nprint("\\n[3/5] import PaddleOCR", flush=True)\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\ntest_path = Path("/content/work/.colab-dev/paddle_smoke_es.png")\ntest_path.parent.mkdir(parents=True, exist_ok=True)\n\nimg = Image.new("RGB", (1200, 300), "white")\ndraw = ImageDraw.Draw(img)\ndraw.text(\n    (60, 90),\n    "Histologia: epitelio plano simple. Prueba OCR en espanol 12345.",\n    fill="black",\n)\nimg.save(test_path)\nprint("Imagen:", test_path, flush=True)\n\nprint("\\n[4/5] inicializar PaddleOCR", flush=True)\nt0 = time.time()\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint(f"Inicializado en {time.time() - t0:.1f} s", flush=True)\n\nprint("\\n[5/5] OCR real", flush=True)\nt0 = time.time()\nresult = ocr.predict(str(test_path))\nelapsed = time.time() - t0\n\ntexts = []\nscores = []\n\nfor res in result:\n    data = getattr(res, "json", res)\n    if callable(data):\n        data = data()\n    if isinstance(data, dict) and "res" in data:\n        data = data["res"]\n    if isinstance(data, dict):\n        texts.extend([str(x) for x in data.get("rec_texts", [])])\n        scores.extend([float(x) for x in data.get("rec_scores", [])])\n\nprint("Tiempo OCR:", f"{elapsed:.2f} s", flush=True)\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\nprint("Scores:", json.dumps(scores), flush=True)\n\nif not texts:\n    raise RuntimeError("PaddleOCR ejecutó pero no devolvió texto.")\n\nprint("\\n✅ SMOKE TEST COMPLETO", flush=True)\n'
worker_path.write_text(WORKER, encoding="utf-8")

env = os.environ.copy()
env.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")
env.setdefault(
    "PADDLE_PDX_CACHE_HOME",
    str(pathlib.Path.home() / ".cache" / "paddlex"),
)
env["PYTHONUNBUFFERED"] = "1"

print("Lanzando worker fresco:")
print(sys.executable, worker_path)
print()

proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"Smoke worker falló con código {rc}")

print("\n✅ El entorno PaddleOCR local funciona sin reiniciar el kernel.")

In [ ]:
import subprocess, sys, pathlib

out = pathlib.Path("/content/work/dev/colab-local/runtime-after-paddle-freeze.txt")
out.parent.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)
out.write_text(result.stdout, encoding="utf-8")
print("Guardado:", out)